In [1]:
from google.colab import drive
import os
import glob
import sys
import torch
import gc
from omegaconf import OmegaConf
import shutil
from pathlib import Path
import wandb

In [2]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    free_mem = torch.cuda.mem_get_info()[0]/1024**2
    print(f"Free GPU memory: {free_mem:.2f} MB")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Free GPU memory: 14807.81 MB


In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
wav_files = glob.glob('/content/drive/MyDrive/Data/**/*.wav', recursive=True)
print(len(wav_files))

1100


In [5]:
%cd /content

if not os.path.exists('vocoders_homework_babushkina'):
    !git clone https://github.com/ksu-bb/vocoders_homework_babushkina.git
    %cd vocoders_homework_babushkina
else:
    %cd vocoders_homework_babushkina
    !git checkout -- .
    !git pull origin main


if '/content/vocoders_homework_babushkina' not in sys.path:
    sys.path.append('/content/vocoders_homework_babushkina')

!pip install -q -r requirements.txt
!apt-get update -qq
!apt-get install -y -qq ffmpeg libsndfile1
!pip install -q audioread soundfile

/content
/content/vocoders_homework_babushkina
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 414 bytes | 414.00 KiB/s, done.
From https://github.com/ksu-bb/vocoders_homework_babushkina
 * branch            main       -> FETCH_HEAD
   91ec9ea..c5d8f47  main       -> origin/main
Updating 91ec9ea..c5d8f47
Fast-forward
 src/train.py | 4 +++-
 1 file changed, 3 insertions(+), 1 deletion(-)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [6]:
COLAB_DATA_PATH = '/content/data/RUSLAN'
os.makedirs('/content/data', exist_ok=True)

if os.path.exists(COLAB_DATA_PATH):
    !rm -rf {COLAB_DATA_PATH}
shutil.copytree('/content/drive/MyDrive/Data', COLAB_DATA_PATH)

wav_count = len(glob.glob(f'{COLAB_DATA_PATH}/**/*.wav', recursive=True))
print(wav_count)

for filepath in glob.glob(f'{COLAB_DATA_PATH}/**/._*', recursive=True):
    try:
        os.remove(filepath)
    except:
        pass

1100


In [7]:
gc.collect()
torch.cuda.empty_cache()
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

config = OmegaConf.load('configs/mel_config.yaml')

config.data.audio_path = '/content/data/RUSLAN'
config.data.filelist_path = '/content/data/filelists'
config.data.batch_size = 1
config.data.num_workers = 0
config.training.gradient_accumulation_steps = 16
config.training.checkpoint_path = '/content/checkpoints'
config.training.epochs = 20

config.generator.upsample_initial_channel = 256

OmegaConf.save(config, 'configs/mel_config.yaml')

In [8]:
!python scripts/prepare_data.py
!wc -l /content/data/filelists/train.txt
!wc -l /content/data/filelists/val.txt

всего файлов: 1100
train: 1045, val: 55
1045 /content/data/filelists/train.txt
55 /content/data/filelists/val.txt


In [9]:
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ksen-kw (ksen-kw-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
%cd /content/vocoders_homework_babushkina

%env PYTORCH_ALLOC_CONF=expandable_segments:True
!PYTHONPATH=/content/vocoders_homework_babushkina python -m src.train

/content/vocoders_homework_babushkina
env: PYTORCH_ALLOC_CONF=expandable_segments:True
Using device: cuda
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ksen-kw (ksen-kw-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ setting up run 70q7994y (0.1s)
wandb: ⣽ setting up run 70q7994y (0.1s)
wandb: ⣾ setting up run 70q7994y (0.1s)
wandb: ⣷ setting up run 70q7994y (0.1s)
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /content/vocoders_homework_babushkina/wandb/run-20260222_154057-70q7994y
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run hifigan_baseline
wandb: ⭐️ View project at https://wandb.ai/ksen-kw-hse-university/ruslan-vocoder
wandb: 🚀 View run at https://wandb.ai/ksen-kw-hse-university/ruslan-vocoder/runs/70q7994y
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.